In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import cv2
import gymnasium as gym
import numpy as np

from collections import deque

#wrapper for env. will crop to 86x86 like in the paper and stack 4 frames, returning a (4, 86, 86)
class BuildState(gym.Wrapper):

    def __init__(self, env, k=4):
        super().__init__(env)
        self.k = k
        self.frames = deque([],maxlen=4)

        self.observation_space = gym.spaces.Box(
            low=0,
            high=255,
            shape=(4, 84, 84),
            dtype=np.uint8 #changed from 32, was too big for buffer
        )

    # convert to greyscale and crop img to a 84x84
    def process_image(self,img):
        gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        resized_img = cv2.resize(gray_img, (84, 110), interpolation=cv2.INTER_AREA)

        # remove from the top, the score board
        cropped_img  = resized_img[18:102, 0:84]
        # normalize for nn
        return cropped_img.astype(np.uint8)


    #start a new game. get inital img, crop and copy it 4 times to fit nn tmplt
    def reset(self,**kwargs):
        observation, info = self.env.reset(**kwargs)
        p_obs = self.process_image(observation)
        #stack k of the img in the Q
        for _ in range(self.k):
            self.frames.append(p_obs)
        #return the Q as a np array
        return np.stack(self.frames, axis=0), info


    '''
    overwrites the step function. will return 4 stack of greysacle 84x84 images
    NOTICE- will aplly the SAME action to all k frames. the reward
    TODO: its unclear how the reward works. sum it up?
    '''
    def step(self, action):
        tottal_reward = 0.0
        # this needed fixing. it skips k frames and saves the k'th. howevert thr queue size is set to 4.
        for _ in range(self.k):
            observation, reward, terminated, truncated, info = self.env.step(action)
            tottal_reward += reward
            #if game ends this is the last frame. dont want to mix new game with old
            if terminated or truncated:
                break

        p_obs = self.process_image(observation)

        #add to top of Que
        self.frames.append(p_obs)

        return np.stack(self.frames, axis=0), np.sign(tottal_reward), terminated, truncated, info



In [3]:
import random
from collections import deque, namedtuple

Frame = namedtuple('frame',('state','action','reward','next_state', 'ended'))

class Replay_buffer:
    def __init__(self,cap):
        self.memory = deque(maxlen=cap)

    #add new fram to buffer
    def push(self, *args):
        self.memory.append(Frame(*args))

    #sample X frames from buffer
    def sample(self,size):
        return random.sample(self.memory,size)

    #allows len to work on this class in other files
    def len(self):
        return len(self.memory)


In [4]:
import torch
import torch.nn.functional as F
import torch.nn as nn
import torch.optim as opt
import numpy as np



class DNQ(nn.Module):
    def __init__(self, output_size=9):
        super(DNQ, self).__init__()
        #conv 4,84,84-> 16, 20,20
        self.conv1 = nn.Conv2d(4,16,kernel_size=8,stride=4)
        #conv 16,20,20-> 32,9,9

        self.conv2 = nn.Conv2d(16,32,kernel_size=4,stride=2)

        self.fc1 = nn.Linear(32 * 9 * 9 ,256)
        self.fc2 = nn.Linear(256 ,output_size)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        #flatten to fit the fc
        x = x.view(x.size(0), -1)

        x = F.relu(self.fc1(x))

        return (self.fc2(x))



class DNQAgent:
    def __init__(self,  device, actions=9, gamma=0.9):

        #setup DNQ
        self.device = device
        self.DNQ = DNQ(actions).to(self.device)
        self.loss_fn = torch.nn.SmoothL1Loss() # gemini is saying SmoothL1Loss is better then mse here. lets see
        # GEMINI thinks i need to use a nigger aplph. will try, not sure what hes absing it on
        self.optimizer= opt.RMSprop(self.DNQ.parameters(), lr=0.00025,alpha=0.95,eps=0.01) #well play with the lr later
        self.gamma = gamma


    # infrence time, finds best action
    def select_action(self, frame):
        state_tensor = torch.from_numpy(np.array(frame)).float().unsqueeze(0).to(self.device) / 255.0
        self.DNQ.eval()
        with torch.no_grad():
            # need to add 'unsqueeze' for cnn to work, adds a dim in satart for batch size
            q_values = self.DNQ(state_tensor)
        self.DNQ.train()

        #pick the acton with highst score
        best_action = q_values.argmax().detach().cpu().item()
        return best_action


    '''
        here is the entire learning process. recives a batch of state from the
        buffer {('state','action','reward','next_state', 'ended'),()...}. ff to find *current* reward for action Si
        then plugges into bellmon eq then fowrd on Si+1 and MSE on the dif
    '''
    def learn_samples(self, batch):
        #extract state, conter to tensor. size: (batch_size,4,84,84). ff-> (batch_size,9) (every action per state)

        states, actions, rewards, next_states, dones = zip(*batch)
        #convert to tensors
        states_t = torch.tensor(np.array(states), dtype=torch.float32).to(self.device) / 255.0
        actions_t = torch.tensor(actions, dtype=torch.int64).unsqueeze(1).to(self.device)
        rewards_t = torch.tensor(rewards, dtype=torch.float32).to(self.device)

        # bounding transformation [-1.0, 1.0]
        rewards_t = torch.clamp(rewards_t, min=-1.0, max=1.0).to(self.device)

        next_states_t = torch.tensor(np.array(next_states), dtype=torch.float32).to(self.device) / 255.0
        dones_t = torch.tensor(dones, dtype=torch.float32).to(self.device)

       # set up the Qs+1 values for bellmon, meaning how much we'd make from next_state
        with torch.no_grad():

            next_q_values = self.DNQ(next_states_t)
            #pick the acton with highst score GEMINI said to add .detach(). see if works#########################
            max_next_q_values = next_q_values.max(1)[0].detach()
            #trick from gemini,if finished it will be only the reward, like the paper
            expected_q_values = rewards_t + (self.gamma * max_next_q_values * (1 - dones_t))


        #pick the matching action to what was done, set as
        q_values = self.DNQ(states_t)

        current_q_values = q_values.gather(1, actions_t).squeeze(1)

        loss = self.loss_fn(current_q_values, expected_q_values)

        self.optimizer.zero_grad()
        loss.backward()

        # ==========================================
        # DIAGNOSTIC: Calculate the L2 Gradient Norm
        # ==========================================
        # total_norm = 0.0
        # for p in self.DNQ.parameters():
        #     if p.grad is not None:
        #         # Calculate the L2 norm of the gradients for this specific layer
        #         param_norm = p.grad.data.norm(2)
        #         # Square it and add to the total sum
        #         total_norm += param_norm.item() ** 2
        #
        # # Take the square root of the total sum
        # total_norm = total_norm ** 0.5
        #
        # # Print the scalar loss and the magnitude of the update
        # print(f"Loss: {loss.item():.4f} | Update Magnitude (L2 Norm): {total_norm:.4f}")
        # ==========================================

        self.optimizer.step()








In [ ]:
#from DNQ_agent import DNQAgent
#from replay_buffer import Replay_buffer
#from build_state import BuildState
####### FOR COLAB
import gc
import ctypes
##########


import matplotlib.pyplot as plt
import gymnasium as gym
import ale_py
import random
import torch
import numpy as np
import os

BUFFER_SIZE = 50000
EPSILON = 0.1
BATCH_SIZE=32
REVIEW_FREQUENCY =25
TOTTAL_EPISODES = 1000

def plot_training_results(q_history, score_history):
    fig, ax1 = plt.subplots(figsize=(10, 5))

    # --- Plot Average Q (Left Axis) ---
    color = 'tab:blue'
    ax1.set_xlabel('Episodes (x10)')
    ax1.set_ylabel('Average Q Value', color=color)
    ax1.plot(q_history, color=color, linewidth=2, label='Avg Q')
    ax1.tick_params(axis='y', labelcolor=color)

    # --- Plot Scores (Right Axis) ---
    ax2 = ax1.twinx()
    color = 'tab:orange'
    ax2.set_ylabel('Score per Episode', color=color)
    ax2.plot(score_history, color=color, alpha=0.3, label='Raw Score')

    # Add a moving average for the score to see the trend through the noise
    if len(score_history) > 10:
        moving_avg = np.convolve(score_history, np.ones(10)/10, mode='valid')
        ax2.plot(moving_avg, color='red', linewidth=1.5, label='Score Trend (MA10)')

    fig.tight_layout()
    plt.title("2013 DQN Training Progress: Breakout")
    plt.show()



def main():
    #setup
    gym.register_envs(ale_py)
    basic_env = gym.make("BreakoutNoFrameskip-v4")
    # "VideoPinballNoFrameskip-v4"  , render_mode="human"

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    env = BuildState(basic_env,k=4) #k - frame skips
    agent = DNQAgent(device=device, actions=4)

    # 1. Define the path to the file you want to load
    load_path = '/content/drive/MyDrive/dl/RL/assin1/dqn_breakout.pth'
   #
    if os.path.exists(load_path):
        print(f"Loading weights from {load_path}...")
        agent.DNQ.load_state_dict(torch.load(load_path, map_location=device, weights_only=True))
        print("Weights loaded successfully.")
    else:
        print(f"No checkpoint found at {load_path}. Starting from scratch.")

    buffer = Replay_buffer(cap=BUFFER_SIZE)

    #evaluation vars-
    tottal_score = 0
    eval_frames = None
    avgQ_history = []
    avg_score_history = []
    step_count = 0
    epsilon = EPSILON
    episode = 0
    ######################FOR COLAB RAM ISSUE
    gc.collect()
    libc = ctypes.CDLL("libc.so.6")
    libc.malloc_trim(0)
    print("Boot-up memory flushed. Starting training...")
    ################################33
    while True:
        #episodes (games-rounds) loop. starts new game

        new_frame,_ = env.reset()
        terminated = False
        truncated = False
        episode_reward = 0

        ended = 0

        while not ended:
            step_count +=1
            #note- this env is a wrapper, the frames are already stacks of k frames
            old_frame = new_frame
            action = env.action_space.sample() if random.random() < epsilon else agent.select_action(old_frame)
            new_frame, reward, terminated, truncated, info = env.step(action)

            tottal_score += reward

            ended = terminated or truncated
            #save to buffer
            buffer.push(old_frame, action, reward, new_frame, ended)
            #builf Q evel when we have enought frames
            if eval_frames is None and buffer.len() > 2000:

                eval_batch = buffer.sample(500)
                states_only, _, _, _, _ = zip(*eval_batch)
                eval_frames = torch.tensor(np.array(states_only), dtype=torch.float32).to(device)
                print("set evaluation set of 500 states captured.")
                gc.collect()
                libc = ctypes.CDLL("libc.so.6")
                libc.malloc_trim(0)
                print(" Q_eval memory flushed. Starting training...")


            if epsilon > 0.05: # 5000 steps in a eps~
                epsilon -= 0.000001

            #now select from buffer and do the learning part
            if buffer.len() > BATCH_SIZE:
                batch = buffer.sample(BATCH_SIZE)# = batch size
                 ######################FOR COLAB RAM ISSUE
                #gc.collect()
                #libc = ctypes.CDLL("libc.so.6")
                #libc.malloc_trim(0)
                ##print("ram test- smaple p1. Starting training...")
               ################################33

                if step_count % 4 == 0: #learn only ever x steps. ram not letting do more.
                  agent.learn_samples(batch)


        episode +=1
        if episode % REVIEW_FREQUENCY == 0 and episode > 0:
            avg_reward = tottal_score/REVIEW_FREQUENCY
            tottal_score = 0
            avg_score_history.append(avg_reward)

            # Q avg eval:
            if eval_frames is not None:
                with torch.no_grad():
                    # 1. Get all Q-values for the 500 states (500, actions)
                    all_q_values = agent.DNQ(eval_frames)

                    # 2. Pick the max Q for each state (the best action the agent sees)
                    max_q_values = all_q_values.max(1)[0]

                    # 3. Average them
                    avg_q = max_q_values.mean().item()
                    avgQ_history.append(avg_q)
            else:
                avg_q = 0.0

            print(f" at episode {episode} avarage reard: {avg_reward} Avg Q: {avg_q:.4f}")
                   # --- AUTO-SAVE CHECKPOINT ---
            # We use a f-string to include the episode number in the filename
            checkpoint_path = f'/content/drive/MyDrive/dl/RL/assin1/dqn_breakout_attempt2.pth'

            torch.save(agent.DNQ.state_dict(), checkpoint_path)

            ##########################FOR COLAB RAM ISSUE
            gc.collect()
            # 2. Force Linux to release the hoarded RAM back to Colab
            libc = ctypes.CDLL("libc.so.6")
            libc.malloc_trim(0)
            #######################33
            if episode >= TOTTAL_EPISODES:
                   break

    env.close()

    plot_training_results(avgQ_history, avg_score_history)

    # observation,_ = env.reset()
    # action = env.action_space.sample()
    # observation, reward, terminated, truncated, _= env.step(action)






if __name__ == '__main__':
    main()











Loading weights from /content/drive/MyDrive/dl/RL/assin1/dqn_breakout.pth...
Weights loaded successfully.
Boot-up memory flushed. Starting training...
set evaluation set of 500 states captured.
 Q_eval memory flushed. Starting training...
